In [ ]:
!pip install -qU langchain-openai langchain-community

In [ ]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("OPENAI_API_KEY loaded")

# 1. 시스템 프롬포트 정의

In [ ]:
SYSTEM_PROMPT = """You are an expert medicine information assistant.

You have access to two tools:

- get_medicine_for_symptom: use this to get medicine information for a specific symptom
- get_user_condition: use this to get the user's condition information

If a user asks about medicine, make sure you consider the user's condition such as pregnancy.
Provide the answer in a clear and structured way."""

# 2. 생성 도구

In [ ]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime

@tool
def get_medicine_for_symptom(symptom: str) -> str:
    """Get medicine information for a given symptom."""
    if symptom == "두통":
        return """
약 이름: 타이레놀
성분: 아세트아미노펜
효능: 두통 완화
주의사항: 임산부 복용 가능

약 이름: 이지엔 이브
성분: 이부프로펜
효능: 통증 및 염증 완화
주의사항: 임산부 복용 금지
"""
    elif symptom == "기침":
        return """
약 이름: 코푸시럽
성분: 덱스트로메토르판
효능: 기침 완화
주의사항: 졸릴 수 있음
"""
    else:
        return "해당 증상에 대한 의약품 정보를 찾을 수 없습니다."

@dataclass
class Context:
    """Custom runtime context schema."""
    user_id: str

@tool
def get_user_condition(runtime: ToolRuntime[Context]) -> str:
    """Retrieve user condition information based on user ID."""
    user_id = runtime.context.user_id

    if user_id == "1":
        return "임산부"
    elif user_id == "2":
        return "성인"
    else:
        return "조건 정보 없음"

# 3. 모델 구성

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "gpt-4o-mini",
    model_provider="openai",
    temperature=0.5,
    max_tokens=1000
)

# 4. 응답 형식 정의

In [ ]:
from dataclasses import dataclass

@dataclass
class ResponseFormat:
    """Response schema for the medicine agent."""
    medicine_response: str
    user_condition: str | None = None

# 5. 메모리 추가

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

# 6. 에이전트 생성 및 실행

In [ ]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_user_condition, get_medicine_for_symptom],
    context_schema=Context,
    response_format=ToolStrategy(ResponseFormat),
    checkpointer=checkpointer
)


config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "두통이 있어요. 먹을 수 있는 약 정보를 알려주세요."
            }
        ]
    },
    config=config,
    context=Context(user_id="1")
)

print(response["structured_response"])


response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "그 약의 성분도 다시 간단히 알려주세요."
            }
        ]
    },
    config=config,
    context=Context(user_id="1")
)

print(response["structured_response"])